# M2: Single Request Runtime

Turn `generate(prompt)` into `Req → Engine → Model`.

```text
prompt → Engine.generate → Req → model.generate() → Req.output_ids → text
```

The code lives in `engine.py`. This notebook uses it and checks how it behaves.

## Setup

`Engine` loads the tokenizer and the model once. The model now stays loaded between requests, which is the first thing a service does differently from a script.

In [ ]:
import threading
import time

import torch

from engine import Engine, EngineBusyError, Req

engine = Engine()
print(engine.device)

## The Request

In M1 the state lived in local variables: `input_ids`, `prompt_len`, the loop counter. Now it lives in a `Req` object.

- `rid`: a unique ID, so the runtime and the client can talk about the same request.
- `input_ids`: the prompt tokens. They never change.
- `output_ids`: the generated tokens, filled in when `model.generate()` returns.
- `max_new_tokens`: a per-request limit.
- `finish_reason`: `None` while running, then `"stop"` (EOS) or `"length"` (limit reached).

The engine samples with the model's default settings from `generation_config`, so the text changes from run to run.

In [ ]:
req = engine.generate("The capital of France is", max_new_tokens=30)

print("rid:          ", req.rid)
print("input_ids:    ", req.input_ids)
print("output_ids:   ", req.output_ids)
print("finish_reason:", req.finish_reason)
print(repr(engine.decode(req)))

In [ ]:
# A short limit ends with "length"; an EOS token ends with "stop"
short = engine.generate("The capital of France is", max_new_tokens=3)
print(short.finish_reason, len(short.output_ids))

## Runtime State and Admission

The `Engine` holds state across calls: `running_req` is the request being served, or `None` when idle.

Capacity is one request. Admission is the check at the door: if a request is already running, the new one is rejected with `EngineBusyError` instead of waiting.

The check is a non-blocking lock. `acquire(blocking=False)` tests and takes the lock in one atomic step, so two threads can never both get in.

To see a rejection, run a long request in a background thread and send a second request while it runs.

In [ ]:
results = {}


def client(name, prompt, max_new_tokens):
    try:
        results[name] = engine.generate(prompt, max_new_tokens=max_new_tokens)
    except EngineBusyError as e:
        results[name] = e


print("before:", engine.running_req)

long_client = threading.Thread(target=client, args=("A", "Write a long story about a dragon.", 100))
long_client.start()
time.sleep(1)  # let A get admitted first

print("during:", engine.running_req.rid)
client("B", "Hello", 5)  # arrives while A is running

long_client.join()
print("after: ", engine.running_req)

for name, result in results.items():
    if isinstance(result, Req):
        print(name, "served:  ", result.finish_reason, len(result.output_ids), "tokens")
    else:
        print(name, "rejected:", result)

`running_req` shows the request in flight. Client B does not wait in a queue. B is rejected at once, and the client must retry later.

## Open Questions

- `generate` uses `lock.acquire(blocking=False)` instead of `if self.running_req is not None`. What goes wrong with the plain check when two threads call `generate` at the same time?
- B was rejected while the GPU was busy with A. What would it take to let B wait instead? (M4)
- `_run` holds the engine for the whole `model.generate()` call. How long is B locked out by a 100-token request?
- `model.generate()` returns only at the end, so the runtime cannot see tokens while they are produced. What does that rule out?